# s08 — Leiden clustering of the FFP output adjacency matrix (MCNS)

MCNS analogue of the FAFB `Data_Processing/s09_FFP_leiden_clustering.ipynb`.

Bipartite Leiden clustering of the FFP -> central-brain output adjacency matrix
produced by `Data_Processing/s07_FFP_ADJ_matrix.m`. Reads from `Processed_Data/`:
- `right_FFP_output_matrix_CB_no_VPN_thr0.csv` — FFP (rows) x post-neuron (cols) synapse matrix
- `right_FFP_table.csv` — FFP neuron table (rows)
- `post_right_FFP_CB_no_VPN_thr0.csv` — post-neuron root_ids (cols)

After clustering, row clusters that contain a single neuron (singletons) and the
post-neuron columns connected only to those singletons are removed, then the
biadjacency is re-aggregated by cluster. Writes the filtered cluster assignments,
inter-cluster connectivity, dendrogram, and the kept/removed row/column bookkeeping
back to `Processed_Data/`.

Requires the `scikit-network` (sknetwork) package.

In [ ]:
from IPython.display import SVG

import os
import numpy as np
from scipy import sparse
import pandas as pd
from sknetwork.visualization import visualize_graph, visualize_bigraph, visualize_dendrogram
from sknetwork.clustering import Leiden, get_modularity
from sknetwork.hierarchy import Paris

# Base folder holding the s07 input matrices and the clustering outputs.
# baseDir (the MCNS root) is resolved from the notebook's location
# (run from MCNS/Data_Processing/), so the code runs after download without editing any path.
baseDir = os.path.dirname(os.getcwd())
processed_dir = os.path.join(baseDir, "Processed_Data")

In [ ]:
# FFP -> central-brain output adjacency matrix (rows = FFP, cols = post neurons)
ADJ = np.loadtxt(os.path.join(processed_dir, "right_FFP_output_matrix_CB_no_VPN_thr0.csv"),
                 delimiter=",", dtype=float)

In [3]:
#ADJ=np.transpose(ADJ)
ADJ.shape

(3904, 16758)

In [4]:
biadjacency = sparse.csr_matrix(ADJ)

In [5]:
def run_leiden_repeatedly(biadjacency, num_iterations, resolution=1):
    best_modularity = -np.inf
    best_labels_row = None
    best_labels_col = None
    best_leiden = None

    for _ in range(num_iterations):
        leiden = Leiden(resolution=resolution, shuffle_nodes=True, sort_clusters=True)
        leiden.fit(biadjacency)

        labels_row = leiden.labels_row_
        labels_col = leiden.labels_col_
        modularity = get_modularity(biadjacency, labels_row, labels_col)

        if modularity > best_modularity:
            best_modularity = modularity
            best_labels_row = labels_row.copy()
            best_labels_col = labels_col.copy()
            best_leiden = leiden

    return best_modularity, best_labels_row, best_labels_col, best_leiden


def remap_labels(labels):
    labels = np.asarray(labels)
    if labels.size == 0:
        return np.array([], dtype=int)

    unique_labels = np.unique(labels)
    mapping = {label: idx for idx, label in enumerate(unique_labels)}
    return np.array([mapping[label] for label in labels], dtype=int)


def aggregate_biadjacency_by_labels(biadjacency, labels_row, labels_col):
    row_labels = np.asarray(labels_row)
    col_labels = np.asarray(labels_col)

    if row_labels.size == 0 or col_labels.size == 0:
        raise ValueError('Filtering removed all rows or all columns. Please check the singleton-row clusters.')

    n_row_clusters = row_labels.max() + 1
    n_col_clusters = col_labels.max() + 1

    row_indicator = sparse.csr_matrix(
        (np.ones(len(row_labels)), (np.arange(len(row_labels)), row_labels)),
        shape=(len(row_labels), n_row_clusters)
    )
    col_indicator = sparse.csr_matrix(
        (np.ones(len(col_labels)), (np.arange(len(col_labels)), col_labels)),
        shape=(len(col_labels), n_col_clusters)
    )

    return row_indicator.T @ biadjacency @ col_indicator



In [ ]:
# Run repeatedly and keep the best partition
num_iterations = 100000
resolution = 3.6
best_modularity, best_labels_row, best_labels_col, best_leiden = run_leiden_repeatedly(
    biadjacency, num_iterations, resolution=resolution
)

print(f"Best Modularity (original): {best_modularity}")

labels_row_unique, counts_row = np.unique(best_labels_row, return_counts=True)
print("Original row cluster sizes:")
print(labels_row_unique, counts_row)

singleton_row_clusters = labels_row_unique[counts_row == 1]
singleton_row_mask = np.isin(best_labels_row, singleton_row_clusters)
row_keep_mask = ~singleton_row_mask

if singleton_row_mask.any():
    connected_cols_mask = np.asarray((biadjacency[singleton_row_mask].sum(axis=0) > 0)).ravel()
else:
    connected_cols_mask = np.zeros(biadjacency.shape[1], dtype=bool)

col_keep_mask = ~connected_cols_mask
filtered_biadjacency = biadjacency[row_keep_mask][:, col_keep_mask].copy()

if filtered_biadjacency.shape[0] == 0 or filtered_biadjacency.shape[1] == 0:
    raise ValueError('After removing singleton-row neurons and their connected columns, no data remains.')

filtered_labels_row = remap_labels(best_labels_row[row_keep_mask])
filtered_labels_col = remap_labels(best_labels_col[col_keep_mask])

print(f"Removed singleton-row neurons: {singleton_row_mask.sum()}")
print(f"Removed connected columns: {connected_cols_mask.sum()}")
print(f"Filtered biadjacency shape: {filtered_biadjacency.shape}")

filtered_labels_row_unique, filtered_counts_row = np.unique(filtered_labels_row, return_counts=True)
print("Filtered row cluster sizes:")
print(filtered_labels_row_unique, filtered_counts_row)

In [7]:
biadjacency_aggregate = aggregate_biadjacency_by_labels(
    filtered_biadjacency,
    filtered_labels_row,
    filtered_labels_col
)



In [8]:
biadjacency_aggregate

<Compressed Sparse Column sparse matrix of dtype 'float64'
	with 893 stored elements and shape (32, 32)>

In [ ]:
# FFP neuron table (rows of ADJ)
Data = pd.read_csv(os.path.join(processed_dir, "right_FFP_table.csv"))

In [10]:
Data

,root_id,type,flywireType,superclass,side,In_Synapse_Optic_R,Out_Synapse_Optic_R,In_Synapse_Central,Out_Synapse_Central,In_Synapse_Optic_L,Out_Synapse_Optic_L,In_Synapse_Total,Out_Synapse_Total,Right_PostPI,Right_PrePI,Left_PostPI,Left_PrePI
0,10012,VS,"VS1,VS2,VS3,VS4,VS5,VS6,VS7,VS8",visual_projection,right,18009,32,127,707,0,0,18136,739,0.985995,-0.913396,-1.0,-1.0
1,10015,HSN,HSN,visual_projection,right,16574,48,166,689,0,0,16740,737,0.980167,-0.869742,-1.0,-1.0
2,10016,HSE,HSE,visual_projection,right,18146,56,181,792,0,0,18327,848,0.980248,-0.867925,-1.0,-1.0
3,10023,HSS,HSS,visual_projection,right,17429,99,254,1139,0,0,17683,1238,0.971272,-0.840065,-1.0,-1.0
4,10025,H2,H2,visual_projection,right,32319,1184,510,3401,0,0,32829,4585,0.968930,-0.483533,-1.0,-1.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3899,546795,LoVP26,LTe65,visual_projection,right,934,0,193,759,0,0,1127,759,0.657498,-1.000000,-1.0,-1.0
3900,549223,LC18,LC18,visual_projection,right,792,171,151,661,0,0,943,832,0.679745,-0.588942,-1.0,-1.0
3901,550055,MeTu3b,MeTu3b,visual_projection,right,530,155,39,199,0,0,569,354,0.862917,-0.124294,-1.0,-1.0
3902,550411,LLPC1,LLPC1,visual_projection,right,429,87,38,183,0,0,467,270,0.837259,-0.355556,-1.0,-1.0


In [11]:
Data_filtered = Data.loc[row_keep_mask].copy()


In [12]:
Data_filtered['Cluster'] = filtered_labels_row


In [ ]:
Data_filtered.to_csv(
    os.path.join(processed_dir, 'leiden_right_FFP_output_CB_no_VPN_thr0_resol3.6_100000_row_singletons_removed.csv'),
    index=False
)

row_id_col = 'root_id' if 'root_id' in Data.columns else Data.columns[0]
row_filter_info = pd.DataFrame({
    'original_index': np.arange(len(Data)),
    'kept': row_keep_mask,
    'removed_singleton_row_cluster': singleton_row_mask,
    'row_cluster_original': best_labels_row,
    'row_id': Data[row_id_col].values
})

row_filter_info.loc[row_filter_info['kept']].to_csv(
    os.path.join(processed_dir, 'leiden_right_FFP_kept_rows_CB_no_VPN_thr0_resol3.6_100000.csv'),
    index=False
)
row_filter_info.loc[~row_filter_info['kept']].to_csv(
    os.path.join(processed_dir, 'leiden_right_FFP_removed_rows_CB_no_VPN_thr0_resol3.6_100000.csv'),
    index=False
)

In [15]:
biadjacency_aggregate_matrix=biadjacency_aggregate.toarray()

In [16]:
biadjacency_aggregate_matrix

array([[1.21690e+05, 6.83000e+02, 5.32000e+02, ..., 5.50000e+02,
        2.60000e+01, 1.90000e+01],
       [5.61000e+02, 6.19730e+04, 9.26000e+02, ..., 2.38500e+03,
        1.62600e+03, 7.00000e+01],
       [4.06000e+02, 1.53000e+02, 1.06946e+05, ..., 2.50000e+01,
        1.42900e+03, 2.22000e+02],
       ...,
       [1.60000e+02, 5.76000e+02, 5.00000e+00, ..., 2.75740e+04,
        1.10000e+01, 5.81000e+02],
       [3.50000e+01, 2.62000e+02, 6.06000e+02, ..., 5.40000e+01,
        1.87030e+04, 8.00000e+00],
       [3.20000e+01, 9.10000e+01, 2.23100e+03, ..., 7.00000e+01,
        7.20000e+01, 4.10200e+03]], shape=(32, 32))

In [17]:
np.shape(biadjacency_aggregate_matrix)

(32, 32)

In [ ]:
np.savetxt(
    os.path.join(processed_dir, 'leiden_right_FFP_intercluster_connectivity_CB_no_VPN_thr0_resol3.6_100000_row_singletons_removed.csv'),
    biadjacency_aggregate_matrix,
    delimiter=','
)

In [19]:
import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix
from sknetwork.hierarchy import Paris
from scipy.cluster.hierarchy import dendrogram
import matplotlib.pyplot as plt

B = biadjacency_aggregate_matrix


In [20]:
np.shape(B)

(32, 32)

In [ ]:
from scipy.sparse import csr_matrix
from sknetwork.hierarchy import Paris
from scipy.cluster.hierarchy import dendrogram
import matplotlib.pyplot as plt
import numpy as np

# --- Raw similarity ---
S = B @ B.T

# Paris (raw)
paris_raw = Paris()
paris_raw.fit(csr_matrix(S))
Z_raw = paris_raw.dendrogram_.astype(float).copy()

# --- Replace Inf/NaN distances with a large finite value ---
mask_bad = ~np.isfinite(Z_raw[:, 2])
if mask_bad.any():
    BIG = 1e12
    Z_raw[mask_bad, 2] = BIG

fig, ax = plt.subplots(1, 1, figsize=(7, 6))
dendrogram(Z_raw, ax=ax, no_labels=True)
ax.set_title('Dendrogram (Raw Similarity, Row Singletons Removed)')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

# Convert to DataFrame
df_raw = pd.DataFrame(Z_raw, columns=['cluster1', 'cluster2', 'distance', 'size'])

# Save to CSV
df_raw.to_csv(
    os.path.join(processed_dir, 'dendrogram_raw_CB_no_VPN_thr0_resol3.6_100000_row_singletons_removed.csv'),
    index=False
)

In [ ]:
# Post-neuron root_ids (columns of ADJ)
PreNeuron = pd.read_csv(os.path.join(processed_dir, "post_right_FFP_CB_no_VPN_thr0.csv"), header=None)
PreNeuron.columns = ['root_id']

In [24]:
PreNeuron_filtered = PreNeuron.loc[col_keep_mask].copy()


In [25]:
PreNeuron_filtered['Cluster'] = filtered_labels_col


In [ ]:
PreNeuron_filtered.to_csv(
    os.path.join(processed_dir, 'leiden_post_right_FFP_CB_no_VPN_thr0_100000_row_singletons_removed.csv'),
    index=False
)

col_filter_info = pd.DataFrame({
    'original_index': np.arange(len(PreNeuron)),
    'kept': col_keep_mask,
    'removed_because_connected_to_singleton_row': connected_cols_mask,
    'col_cluster_original': best_labels_col,
    'root_id': PreNeuron.iloc[:, 0].values
})

col_filter_info.loc[col_filter_info['kept']].to_csv(
    os.path.join(processed_dir, 'leiden_post_right_FFP_kept_cols_CB_no_VPN_thr0_100000.csv'),
    index=False
)
col_filter_info.loc[~col_filter_info['kept']].to_csv(
    os.path.join(processed_dir, 'leiden_post_right_FFP_removed_cols_CB_no_VPN_thr0_100000.csv'),
    index=False
)

In [27]:
get_modularity(filtered_biadjacency, labels=filtered_labels_row, labels_col=filtered_labels_col)


np.float64(0.7101746509586352)